# SupportIQ Minimal RAG Experiment

This notebook prototypes the AI workflow before it is converted into tested Python modules.

Current stage: load the fictional Markdown knowledge base while preserving source metadata for future chunks and citations.

## 1. Imports and paths

Only the Python standard library is required for document loading.

In [ ]:
from dataclasses import dataclass
from datetime import date
from pathlib import Path

KNOWLEDGE_BASE = Path("../data/sample_knowledge_base")
REQUIRED_METADATA = {
    "document_id",
    "title",
    "version",
    "last_updated",
    "product_area",
}

## 2. Document representation

`SupportDocument` keeps the source identity attached to the document text. Later, chunks will inherit this metadata so retrieval results can produce citations.

In [ ]:
@dataclass(frozen=True)
class SupportDocument:
    document_id: str
    title: str
    version: str
    last_updated: date
    product_area: str
    source_path: Path
    content: str

## 3. Parse one Markdown document

The parser separates the front matter from the searchable content and fails clearly when a document is malformed. It intentionally supports only the simple metadata format used by this experiment.

In [ ]:
def parse_document(path: Path) -> SupportDocument:
    text = path.read_text(encoding="utf-8")
    lines = text.splitlines()

    if not lines or lines[0] != "---":
        raise ValueError(f"{path} must start with a front-matter delimiter")

    try:
        closing_delimiter = lines.index("---", 1)
    except ValueError as exc:
        raise ValueError(f"{path} is missing the closing front-matter delimiter") from exc

    metadata: dict[str, str] = {}
    for line in lines[1:closing_delimiter]:
        key, separator, value = line.partition(":")
        key = key.strip()
        value = value.strip()

        if not separator or not key or not value:
            raise ValueError(f"{path} contains invalid metadata: {line!r}")
        if key in metadata:
            raise ValueError(f"{path} contains duplicate metadata key: {key}")
        metadata[key] = value

    missing_fields = REQUIRED_METADATA - metadata.keys()
    if missing_fields:
        missing = ", ".join(sorted(missing_fields))
        raise ValueError(f"{path} is missing required metadata: {missing}")

    content = "\n".join(lines[closing_delimiter + 1 :]).strip()
    if not content:
        raise ValueError(f"{path} contains no document content")

    try:
        last_updated = date.fromisoformat(metadata["last_updated"])
    except ValueError as exc:
        raise ValueError(f"{path} has an invalid last_updated date") from exc

    return SupportDocument(
        document_id=metadata["document_id"],
        title=metadata["title"],
        version=metadata["version"],
        last_updated=last_updated,
        product_area=metadata["product_area"],
        source_path=path,
        content=content,
    )

## 4. Load the complete knowledge base

Files are sorted to make repeated runs deterministic. Duplicate document IDs are rejected because they would make citations ambiguous.

In [ ]:
def load_documents(directory: Path) -> list[SupportDocument]:
    if not directory.is_dir():
        raise ValueError(f"Knowledge-base directory does not exist: {directory}")

    paths = sorted(directory.glob("*.md"))
    if not paths:
        raise ValueError(f"Knowledge-base directory contains no Markdown files: {directory}")

    documents = [parse_document(path) for path in paths]
    document_ids = [document.document_id for document in documents]

    if len(set(document_ids)) != len(document_ids):
        raise ValueError("Knowledge base contains duplicate document IDs")

    return documents

## 5. Load and inspect the documents

Run this cell and confirm that all expected source IDs, titles, and paths are visible before implementing chunking.

In [ ]:
documents = load_documents(KNOWLEDGE_BASE)

print(f"Loaded {len(documents)} support documents:")
for document in documents:
    print(
        f"- {document.document_id}: {document.title} "
        f"[{document.product_area}] ({document.source_path})"
    )

## 6. Experiment checks

These assertions provide immediate notebook feedback. They are not a replacement for pytest tests after the experiment is converted into Python modules.

In [ ]:
assert len(documents) == 10
assert len({document.document_id for document in documents}) == 10
assert all(document.title for document in documents)
assert all(document.content for document in documents)
assert all(document.source_path.exists() for document in documents)
assert all(not document.content.startswith("---") for document in documents)

print("Document-loading checks passed.")

## Next stage

Inspect several complete documents, then design one simple chunking approach. Do not add embeddings until chunk boundaries have been reviewed.